# 3. Entrenamiento de Modelos

Este notebook implementa el pipeline de entrenamiento para la detección de exoplanetas.

## Estructura

1. **Cargar datos**: Leer los CSVs preprocesados
2. **DataLoaders**: Crear datasets y dataloaders de PyTorch
3. **Definir modelo**: LSTM básico para empezar
4. **Training loop**: Con early stopping y métricas por época
5. **Evaluación**: Métricas finales en test

## 3.1 Imports y configuración

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
import time
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Configuración
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Directorios
DATA_DIR = Path("data/processed")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

print("✓ Librerías cargadas")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jorvarea/Desktop/Universidad/Tercer_curso/primer_cuatrimestre/Deep_Learning/practicas/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/jorvarea/Desktop/Universidad/Tercer_curso/primer_cuatrimestre/Deep_Learning/practicas/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start(

Device: cpu
✓ Librerías cargadas


## 3.2 Cargar datos

In [3]:
# Cargar CSVs
print("Cargando datos...")
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

print(f"✓ Train: {len(df_train):,} samples")
print(f"✓ Val: {len(df_val):,} samples")
print(f"✓ Test: {len(df_test):,} samples")

# Identificar columnas de features
global_cols = [c for c in df_train.columns if c.startswith('global_')]
local_cols = [c for c in df_train.columns if c.startswith('local_')]

print(f"\nFeatures:")
print(f"  - Global view: {len(global_cols)} features")
print(f"  - Local view: {len(local_cols)} features")

# Distribución de clases
print(f"\nDistribución de clases (Train):")
print(df_train['av_training_set'].value_counts())
n_pos = df_train['label'].sum()
n_neg = len(df_train) - n_pos
print(f"  Ratio desbalance: 1:{n_neg/n_pos:.2f}")

Cargando datos...
✓ Train: 12,589 samples
✓ Val: 1,574 samples
✓ Test: 1,574 samples

Features:
  - Global view: 2001 features
  - Local view: 201 features

Distribución de clases (Train):
av_training_set
AFP    7643
PC     2885
NTP    2061
Name: count, dtype: int64
  Ratio desbalance: 1:3.36


## 3.3 Dataset y DataLoaders

In [2]:
class ExoplanetDataset(Dataset):
    """
    Dataset para curvas de luz de Kepler.
    Puede usar global_view, local_view o ambas.
    """
    def __init__(self, df: pd.DataFrame, feature_cols: list[str]):
        self.features = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)
    
    def __len__(self) -> int:
        return len(self.labels)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.labels[idx]


def create_dataloaders(
    df_train: pd.DataFrame, 
    df_val: pd.DataFrame, 
    df_test: pd.DataFrame,
    feature_cols: list[str],
    batch_size: int = 32
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """
    Crea DataLoaders para train, val y test.
    """
    train_dataset = ExoplanetDataset(df_train, feature_cols)
    val_dataset = ExoplanetDataset(df_val, feature_cols)
    test_dataset = ExoplanetDataset(df_test, feature_cols)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader


print("✓ Dataset class definida")

✓ Dataset class definida


## 3.4 Funciones de entrenamiento

In [5]:
def train_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device
) -> tuple[float, float]:
    """
    Entrena una época.
    Returns: (loss, accuracy)
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)
    
    return total_loss / total, correct / total


def evaluate(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module,
    device: torch.device
) -> tuple[float, float]:
    """
    Evalúa el modelo en un loader.
    Returns: (loss, accuracy)
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)
    
    return total_loss / total, correct / total


print("✓ Funciones train_epoch y evaluate definidas")

✓ Funciones train_epoch y evaluate definidas


In [6]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    n_epochs: int = 50,
    patience: int = 5,
    verbose: bool = True
) -> dict:
    """
    Entrena el modelo con early stopping.
    
    Args:
        model: Modelo a entrenar
        train_loader: DataLoader de entrenamiento
        val_loader: DataLoader de validación
        criterion: Función de pérdida
        optimizer: Optimizador
        device: Device (cuda/cpu)
        n_epochs: Número máximo de épocas
        patience: Épocas sin mejora antes de parar
        verbose: Imprimir progreso
    
    Returns:
        dict con historial de entrenamiento
    """
    model = model.to(device)
    
    # Historial
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'epoch_time': [],
    }
    
    # Early stopping
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    print(f"Iniciando entrenamiento (max {n_epochs} épocas, patience={patience})")
    print("-" * 70)
    
    for epoch in range(n_epochs):
        epoch_start = time.time()
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        epoch_time = time.time() - epoch_start
        
        # Guardar historial
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(epoch_time)
        
        # Print progress
        if verbose:
            print(f"Epoch {epoch+1:3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | "
                  f"Time: {epoch_time:.1f}s")
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⚠ Early stopping en época {epoch+1} (sin mejora en {patience} épocas)")
                break
    
    # Restaurar mejor modelo
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✓ Restaurado mejor modelo (val_loss={best_val_loss:.4f})")
    
    # Resumen
    total_time = sum(history['epoch_time'])
    avg_time = np.mean(history['epoch_time'])
    actual_epochs = len(history['train_loss'])
    
    print("-" * 70)
    print(f"Entrenamiento completado:")
    print(f"  - Épocas: {actual_epochs}")
    print(f"  - Tiempo total: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"  - Tiempo promedio por época: {avg_time:.1f}s")
    print(f"  - Mejor val_loss: {best_val_loss:.4f}")
    
    history['best_val_loss'] = best_val_loss
    history['actual_epochs'] = actual_epochs
    history['total_time'] = total_time
    history['avg_epoch_time'] = avg_time
    
    return history


print("✓ Función train_model definida")

✓ Función train_model definida


## 3.5 Funciones de evaluación en test

In [ ]:
def evaluate_model(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device
) -> dict:
    """
    Evaluación completa del modelo en test set.
    Calcula múltiples métricas relevantes para problema desbalanceado.
    """
    model.eval()
    model = model.to(device)
    
    all_probs = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            
            outputs = model(inputs)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Métricas
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    
    # AUC-ROC y AUC-PR (importantes para desbalance)
    auc_roc = roc_auc_score(all_labels, all_probs)
    auc_pr = average_precision_score(all_labels, all_probs)
    
    results = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'probs': all_probs,
        'preds': all_preds,
        'labels': all_labels
    }
    
    return results

# 4. Modelos

## 4.1 LSTM 

In [12]:
class SimpleLSTM(nn.Module):
    """
    LSTM simple para clasificación de curvas de luz.
    
    La entrada es una secuencia 1D (global_view de 2001 puntos).
    Se trata cada punto como un timestep con 1 feature.
    """
    def __init__(
        self, 
        input_size: int = 1,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
        bidirectional: bool = False
    ):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Output layer
        fc_input_size = hidden_size * self.num_directions
        self.fc = nn.Linear(fc_input_size, 1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len) -> (batch, seq_len, 1)
        x = x.unsqueeze(-1)
        
        # LSTM forward
        # output: (batch, seq_len, hidden_size * num_directions)
        # hidden: (num_layers * num_directions, batch, hidden_size)
        output, (hidden, cell) = self.lstm(x)
        
        # Tomar el último hidden state
        if self.bidirectional:
            # Concatenar forward y backward
            hidden_forward = hidden[-2, :, :]  # (batch, hidden_size)
            hidden_backward = hidden[-1, :, :]
            hidden_cat = torch.cat([hidden_forward, hidden_backward], dim=1)
        else:
            hidden_cat = hidden[-1, :, :]  # (batch, hidden_size)
        
        # Dropout + FC
        out = self.dropout(hidden_cat)
        out = self.fc(out)
        
        return out.squeeze(-1)  # (batch,)


print("✓ Modelo SimpleLSTM definido")

✓ Modelo SimpleLSTM definido


# 5. Configuración

In [8]:
# ============================================================
# HIPERPARÁMETROS
# ============================================================

# Datos
BATCH_SIZE = 32
USE_GLOBAL = True   # Usar global_view (2001 features)
USE_LOCAL = False   # Usar local_view (201 features)

# Modelo
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.3
BIDIRECTIONAL = False

# Training
LEARNING_RATE = 0.001
N_EPOCHS = 2
PATIENCE = 1

# ============================================================

# Seleccionar features
feature_cols = []
if USE_GLOBAL:
    feature_cols.extend(global_cols)
if USE_LOCAL:
    feature_cols.extend(local_cols)

print(f"Features seleccionadas: {len(feature_cols)}")
print(f"  - Global: {USE_GLOBAL} ({len(global_cols)} features)")
print(f"  - Local: {USE_LOCAL} ({len(local_cols)} features)")

Features seleccionadas: 2001
  - Global: True (2001 features)
  - Local: False (201 features)


# 6. Entrenamiento

In [9]:
# Crear DataLoaders
train_loader, val_loader, test_loader = create_dataloaders(
    df_train, df_val, df_test, 
    feature_cols, 
    batch_size=BATCH_SIZE
)

print(f"DataLoaders creados:")
print(f"  - Train: {len(train_loader)} batches")
print(f"  - Val: {len(val_loader)} batches")
print(f"  - Test: {len(test_loader)} batches")

# Crear modelo
model = SimpleLSTM(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    bidirectional=BIDIRECTIONAL
)

# Contar parámetros
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModelo: SimpleLSTM")
print(f"  - Hidden size: {HIDDEN_SIZE}")
print(f"  - Num layers: {NUM_LAYERS}")
print(f"  - Bidirectional: {BIDIRECTIONAL}")
print(f"  - Parámetros: {n_params:,}")

DataLoaders creados:
  - Train: 394 batches
  - Val: 50 batches
  - Test: 50 batches

Modelo: SimpleLSTM
  - Hidden size: 64
  - Num layers: 2
  - Bidirectional: False
  - Parámetros: 50,497


In [10]:
# Criterion y optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Entrenar
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    n_epochs=N_EPOCHS,
    patience=PATIENCE,
    verbose=True
)

Iniciando entrenamiento (max 2 épocas, patience=1)
----------------------------------------------------------------------


KeyboardInterrupt: 

# 7. Evaluación en Test

In [ ]:
# Evaluar en test
print("Evaluando en Test set...")
test_results = evaluate_model(model, test_loader, device)

print("\n" + "="*50)
print("RESULTADOS EN TEST")
print("="*50)
print(f"Accuracy:  {test_results['accuracy']*100:.2f}%")
print(f"Precision: {test_results['precision']*100:.2f}%")
print(f"Recall:    {test_results['recall']*100:.2f}%")
print(f"F1-Score:  {test_results['f1']*100:.2f}%")
print(f"AUC-ROC:   {test_results['auc_roc']:.4f}")
print(f"AUC-PR:    {test_results['auc_pr']:.4f}")

print("\nClassification Report:")
print(classification_report(
    test_results['labels'], 
    test_results['preds'],
    target_names=['No Planeta (0)', 'Planeta (1)']
))

# 8. Visualización del entrenamiento

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss por época')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-', label='Train')
axes[1].plot(epochs, [a*100 for a in history['val_acc']], 'r-', label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy por época')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Tiempo por época
axes[2].bar(epochs, history['epoch_time'], color='steelblue')
axes[2].axhline(y=history['avg_epoch_time'], color='red', linestyle='--', label=f"Avg: {history['avg_epoch_time']:.1f}s")
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Tiempo (s)')
axes[2].set_title('Tiempo por época')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 9. Guardar modelo y resultados

In [ ]:
# ============================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================
EXPERIMENT_NAME = "lstm_baseline"  # Cambiar para cada experimento
LOSS_TYPE = "BCE"  # BCE, WBCE, FocalLoss

# ============================================================

# Guardar modelo
model_path = MODELS_DIR / f"{EXPERIMENT_NAME}.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'hyperparameters': {
        'hidden_size': HIDDEN_SIZE,
        'num_layers': NUM_LAYERS,
        'dropout': DROPOUT,
        'bidirectional': BIDIRECTIONAL,
    },
    'history': history,
    'test_results': {k: v for k, v in test_results.items() if k not in ['probs', 'preds', 'labels']},
}, model_path)

print(f"✓ Modelo guardado en: {model_path}")

# ============================================================
# GUARDAR RESULTADOS EN CSV
# ============================================================
import os
from datetime import datetime

RESULTS_FILE = DATA_DIR.parent / "results.csv"

# Crear registro del experimento
experiment_results = {
    'experiment_name': EXPERIMENT_NAME,
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'model_type': 'LSTM',
    'loss_type': LOSS_TYPE,
    'use_global': USE_GLOBAL,
    'use_local': USE_LOCAL,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'bidirectional': BIDIRECTIONAL,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'max_epochs': N_EPOCHS,
    'patience': PATIENCE,
    'actual_epochs': history['actual_epochs'],
    'total_time_sec': history['total_time'],
    'avg_epoch_time_sec': history['avg_epoch_time'],
    'best_val_loss': history['best_val_loss'],
    'test_accuracy': test_results['accuracy'],
    'test_precision': test_results['precision'],
    'test_recall': test_results['recall'],
    'test_f1': test_results['f1'],
    'test_auc_roc': test_results['auc_roc'],
    'test_auc_pr': test_results['auc_pr'],
}

# Guardar en CSV (append si existe)
results_df = pd.DataFrame([experiment_results])

if RESULTS_FILE.exists():
    existing_df = pd.read_csv(RESULTS_FILE)
    # Eliminar experimento previo con mismo nombre si existe
    existing_df = existing_df[existing_df['experiment_name'] != EXPERIMENT_NAME]
    results_df = pd.concat([existing_df, results_df], ignore_index=True)

results_df.to_csv(RESULTS_FILE, index=False)
print(f"✓ Resultados guardados en: {RESULTS_FILE}")

# ============================================================
# RESUMEN FINAL
# ============================================================
print("\n" + "="*50)
print("RESUMEN FINAL")
print("="*50)
print(f"Experimento: {EXPERIMENT_NAME}")
print(f"Modelo: LSTM ({'BiLSTM' if BIDIRECTIONAL else 'Unidireccional'})")
print(f"Loss: {LOSS_TYPE}")
print(f"Features: {'global' if USE_GLOBAL else ''} {'+ local' if USE_LOCAL else ''}")
print(f"Épocas: {history['actual_epochs']}/{N_EPOCHS}")
print(f"Tiempo total: {history['total_time']:.1f}s ({history['total_time']/60:.1f} min)")
print(f"Tiempo promedio/época: {history['avg_epoch_time']:.1f}s")
print(f"\nMétricas en Test:")
print(f"  - Accuracy:  {test_results['accuracy']*100:.2f}%")
print(f"  - Precision: {test_results['precision']*100:.2f}%")
print(f"  - Recall:    {test_results['recall']*100:.2f}%")
print(f"  - F1-Score:  {test_results['f1']*100:.2f}%")
print(f"  - AUC-ROC:   {test_results['auc_roc']:.4f}")
print(f"  - AUC-PR:    {test_results['auc_pr']:.4f}")